# Task 1：植物分类 CNN
一个文件完成数据读取、训练、验证和生成 `submission_cnn.csv`。

- 数据：500 张训练图，5 类各 100 张；378 张测试图。
- 方法：自定义 CNN，加入残差跳跃连接（skip addition）和多层特征拼接（concat），从随机权重开始训练。
- 划分：分层划分 80% 训练 / 20% 验证，保存验证分数最高的模型。
- 指标：每张图只有一个类别，因此 **micro-F1 等于准确率**。

**Kaggle 用法：**导入本 Notebook，添加比赛数据，开启 GPU，然后从上到下运行。不需要下载预训练权重，可以关闭 Internet。输出位于 `/kaggle/working/`，提交其中的 `submission_cnn.csv`。

本地运行需要 `torch torchvision pandas scikit-learn pillow tqdm`，从项目根目录或 `task1` 目录打开 Notebook 即可。数据文件夹中的 `task2` 是下载包原有名称，无需修改。

这一版保存为 `best_cnn.pth` 和 `submission_cnn.csv`，方便保留之前的模型结果。**修改训练参数后请重启内核并运行全部单元格，不要只重复运行训练单元格。**


## 1. 导入依赖与设置参数
通常只需要调整下面的训练轮数、批大小和学习率。显存不足时把 `BATCH_SIZE` 改为 16。


In [ ]:
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch import nn
from torch.utils.data import DataLoader, Dataset, Subset
from torchvision import datasets, transforms
from tqdm.auto import tqdm

SEED = 42
EPOCHS = 50
BATCH_SIZE = 32
LR = 1e-3

# Kaggle 自动使用挂载数据；本地从当前目录向下查找。
INPUT_ROOT = Path("/kaggle/input") if Path("/kaggle/input").exists() else Path.cwd()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd() / "output"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
NUM_WORKERS = 0  # Notebook 使用单进程读取，避免 DataLoader 子进程清理报错。

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cpu":
    print("当前使用 CPU；在 Kaggle 设置中开启 GPU 可以加快训练。")


## 2. 找到数据与提交模板
训练图按文件夹名标注类别。测试图的读取顺序使用模板中的 `ID`，最终提交植物名称。


In [ ]:
data_dirs = sorted(
    path.parent for path in INPUT_ROOT.rglob("train")
    if path.is_dir() and (path.parent / "test").is_dir()
)
sample_files = sorted(INPUT_ROOT.rglob("submission-for-task2.csv"))
if len(data_dirs) != 1 or len(sample_files) != 1:
    raise ValueError("请把 INPUT_ROOT 指向本比赛的数据目录，确保只有一套 train/test 和提交模板。")

DATA_DIR = data_dirs[0]
submission = pd.read_csv(sample_files[0])
assert submission.columns.tolist() == ["ID", "Category"], "提交模板应包含 ID,Category 两列"
assert submission["ID"].is_unique, "提交模板中存在重复 ID"
assert all((DATA_DIR / "test" / name).is_file() for name in submission["ID"]), "部分测试图片不存在"

print("Data:", DATA_DIR)
print("Test images:", len(submission))
submission.head()


## 3. 数据增强与分层划分
训练时随机裁剪、翻转、旋转并轻微调整颜色；验证和测试使用固定预处理。验证集独立保留，不参与训练。


In [ ]:
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(180),
    transforms.ColorJitter(brightness=0.15, contrast=0.15, saturation=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])
eval_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

# 分别创建两个 ImageFolder，防止训练增强影响验证集。
train_data = datasets.ImageFolder(DATA_DIR / "train", transform=train_transform)
val_data = datasets.ImageFolder(DATA_DIR / "train", transform=eval_transform)
classes = train_data.classes
train_indices, val_indices = train_test_split(
    np.arange(len(train_data)), test_size=0.2, random_state=SEED,
    stratify=train_data.targets,
)
val_labels = np.array(train_data.targets)[val_indices]


class TestDataset(Dataset):
    def __init__(self, ids):
        self.paths = [DATA_DIR / "test" / name for name in ids]

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, index):
        with Image.open(self.paths[index]) as image:
            image = eval_transform(image.convert("RGB"))
        return image, -1  # 测试集没有标签，使用占位值。


loader_options = dict(
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS,
    pin_memory=(device.type == "cuda"),
)
train_loader = DataLoader(Subset(train_data, train_indices), shuffle=True, **loader_options)
val_loader = DataLoader(Subset(val_data, val_indices), shuffle=False, **loader_options)
test_loader = DataLoader(TestDataset(submission["ID"]), shuffle=False, **loader_options)

print("Classes:", classes)
print(f"Train: {len(train_indices)} | Validation: {len(val_indices)}")
print(pd.Series([classes[i] for i in train_data.targets]).value_counts().sort_index())


## 4. 自定义 CNN：Skip + Concat
网络按顺序提取 3 个阶段的特征，每个阶段包含 2 个残差块：

`输入 → Stem → Stage 1（64 通道）→ Stage 2（128 通道）→ Stage 3（256 通道）`

- **Skip：**每个残差块把卷积分支和输入旁路相加。如果尺寸或通道数变化，旁路用一个 1×1 卷积对齐。
- **Concat：**对三个阶段的输出分别做全局平均池化，再拼成 `64 + 128 + 256 = 448` 维特征用于分类。这样不必把不同大小的特征图强行拼接。

模型从随机权重开始训练；GPU 使用混合精度减少显存占用。


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.main = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        self.shortcut = nn.Identity()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x):
        return self.relu(self.main(x) + self.shortcut(x))  # Skip：逐元素相加。


class SkipConcatCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, stride=2, padding=1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
        )
        self.stage1 = nn.Sequential(ResidualBlock(32, 64, 2), ResidualBlock(64, 64))
        self.stage2 = nn.Sequential(ResidualBlock(64, 128, 2), ResidualBlock(128, 128))
        self.stage3 = nn.Sequential(ResidualBlock(128, 256, 2), ResidualBlock(256, 256))
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = nn.Sequential(nn.Dropout(0.3), nn.Linear(64 + 128 + 256, num_classes))

    def forward(self, x):
        x = self.stem(x)
        x1 = self.stage1(x)  # [B, 64, 56, 56]，输入为 224×224。
        x2 = self.stage2(x1)  # [B, 128, 28, 28]
        x3 = self.stage3(x2)  # [B, 256, 14, 14]
        features = [self.pool(feature).flatten(1) for feature in (x1, x2, x3)]
        x = torch.cat(features, dim=1)  # Concat：沿特征维拼接，得到 [B, 448]。
        return self.classifier(x)


model = SkipConcatCNN(num_classes=len(classes)).to(device)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))


@torch.no_grad()
def predict(loader):
    model.eval()
    predictions = []
    for images, _ in loader:
        images = images.to(device, non_blocking=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            logits = model(images)
        predictions.extend(logits.argmax(dim=1).cpu().tolist())
    return np.array(predictions)


## 5. 训练并保存验证集上的最佳模型
每轮输出训练损失和验证 micro-F1。分数来自 100 张验证图片，是本地验证结果，不代表 Kaggle 排名。


In [ ]:
best_f1 = -1.0
checkpoint_path = OUTPUT_DIR / "best_cnn.pth"
history = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total_loss = 0.0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}", leave=False):
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, enabled=(device.type == "cuda")):
            loss = criterion(model(images), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item() * images.size(0)

    train_loss = total_loss / len(train_indices)
    val_f1 = float((predict(val_loader) == val_labels).mean())
    history.append({"epoch": epoch, "train_loss": train_loss, "val_micro_f1": val_f1})
    print(f"Epoch {epoch:02d} | loss={train_loss:.4f} | val_micro_f1={val_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        torch.save({
            "model": model.state_dict(), "classes": classes,
            "val_micro_f1": val_f1, "epoch": epoch,
        }, checkpoint_path)
    scheduler.step()

print(f"Best validation micro-F1: {best_f1:.4f}")
pd.DataFrame(history)


## 6. 生成 Kaggle 提交文件
加载验证集分数最高的权重，只替换模板的 `Category`，保留原始 `ID` 顺序，不输出额外索引列。


In [ ]:
checkpoint = torch.load(checkpoint_path, map_location=device, weights_only=True)
assert checkpoint["classes"] == classes, "模型与当前数据的类别顺序不一致"
model.load_state_dict(checkpoint["model"])

test_predictions = predict(test_loader)
submission["Category"] = [classes[index] for index in test_predictions]
assert len(submission) == len(test_loader.dataset)
assert submission["Category"].isin(classes).all()
assert not submission.isna().any().any()

submission_path = OUTPUT_DIR / "submission_cnn.csv"
submission.to_csv(submission_path, index=False)
print(f"使用第 {checkpoint['epoch']} 轮模型，验证 micro-F1={checkpoint['val_micro_f1']:.4f}")
print("Saved:", submission_path.resolve())
print(submission["Category"].value_counts())
submission.head()
